# World of Shadow Work — **v2 "Move 1": DREAMS** (Colab Pro+, A100 80GB)

> Companion to `reagency/factory/wosw_colab.ipynb`. This notebook does the **dream bake**: LoRA-train a
> diffusion model on the existing museum corpus, generate three kinds of archival **DREAMS**, CLIP-embed
> them with the galaxy's *exact* model, fold them into the embeddings as `type=2` points, and **re-bake**
> the galaxy. The committed 50k galaxy is **superseded** by the re-baked combined set.

Run on an **A100 80GB** runtime (Runtime → Change runtime type → A100 → **High-RAM**), then **Run all**.
FLUX.1-dev 12B needs the 80GB card; SDXL fits ~16GB and is the validation pass.

**BAKE DAG (honored top→bottom):**
```
caption → LoRA train → generate dreams (PNGs) → CLIP-embed dreams (galaxy's model)
   → concatenate with the EXISTING work/embeddings.npy + extend meta (type='dream'→points.bin type=2, provenance)
   → re-run stage_b_layout.py on the COMBINED set → new assets that SUPERSEDE the galaxy
```
Cluster/hero indices shift after the merge — that is **accepted** by design.

**CRITICAL — embedding-space identity.** The installed galaxy is `embed_dim=1024`
(`reagency/assets/manifest.json` line 6), produced by open_clip `xlm-roberta-large-ViT-H-14` /
`frozen_laion5b_s13b_b90k` (`wosw_colab.ipynb` config, `MULTILINGUAL=True`). Dreams **must** be embedded
with that *identical* model or they land in a different space and the merge is meaningless. The diffusion
LoRA model (SDXL/FLUX) only *generates* the dream PNGs; CLIP only *embeds* them for placement. They are
separate — the CONFIG cell locks `CLIP_MODEL`/`CLIP_PRETRAINED` apart from the diffusion `MODEL` flag.

**Two .jpg-vs-.png facts that bite:**
- `stage_a_embed.py:59` globs `corpus/images/**/*.jpg` ONLY. We DELIBERATELY keep dreams OUT of
  `corpus/` — saving them into `corpus/images/dreams/` would (a) be dropped if PNG, and (b) insert
  `dreams` alphabetically BETWEEN `agency` and `labor`, shifting every later corpus index and breaking
  provenance. Instead dreams live in `work/dreams/*.png`, are embedded SEPARATELY, and **appended** after
  all 50k corpus rows so the original order/ids are preserved (GROUND merge strategy).
- `stage_b_layout.py:267` emits only `type` 0/1. We patch it in-notebook so `type='dream'`→**2**.

**Three dream ops (Move 1), scoped to hero pairs/images:**
1. img2img latent-blend **dream-melt** between two archive photos (strength ~0.62). Provenance = both.
2. rembg/U2Net **shadow inpaint** — inpaint the segmented-out subject (the *disowned*). Provenance = one.
3. edge **outpaint** — extend the archive past its frame (inpaint an outer ring). Provenance = one.

Every dream records a **provenance** row (`dream_id → [source corpus indices / source_ids]`) so the allolib
runtime's HumanTrace can later surface the *real* source artworks.

Reuses, unchanged-in-invocation, the existing factory: `stage_a_embed.py`, `stage_b_layout.py`,
`fetch_corpus.py`, `corpus/ATTRIBUTION.csv` (11-col schema), `assets/manifest.json`. This notebook adds
only the dream stages; everything else mirrors `wosw_colab.ipynb`.

**Checkpoints.** Every expensive stage (captions, LoRA weights, dream PNGs, dream + merged embeddings)
is mirrored to a Drive checkpoint dir. The `RUN_*` toggles skip stages that already produced a checkpoint.

> **Run order:** do the **SDXL validate pass** end-to-end first (`MODEL="sdxl"`), confirm the bake
> produces `type=2` points, THEN flip `MODEL="flux"` (+ `HF_TOKEN`) for the final high-fidelity bake.
> See the run-order notes at the bottom.

## CONFIG

The single source of truth for the whole notebook. Mirrors `wosw_colab.ipynb` cell 1 and adds the
diffusion / LoRA / dream / checkpoint flags. **Edit only this cell** between the two passes (set
`MODEL`, `HF_TOKEN`).

In [ ]:
# ===== CONFIG (Move 1 — DREAMS) =====  mirrors wosw_colab.ipynb cell 1; adds diffusion/LoRA/dream flags.
# ONE source of truth for the whole notebook. Set MODEL, run the two-pass workflow (see runOrderNotes).
import os

# --- which diffusion base to LoRA-train + dream with THIS pass ---
MODEL = "sdxl"          # "sdxl" = VALIDATE pass (LoRA <1hr, ~16GB) | "flux" = FINAL FLUX.1-dev 12B bake
assert MODEL in ("sdxl", "flux"), MODEL
MODEL_ID = {"sdxl": "stabilityai/stable-diffusion-xl-base-1.0",
            "flux": "black-forest-labs/FLUX.1-dev"}[MODEL]

# --- repo / push (same as wosw_colab cell 1) ---
GH_TOKEN = ""           # GitHub token (repo scope) to push the re-baked assets. Blank => download tarball.
REPO     = "9LiveZZZ-Git/MAT201B_Projects"
BRANCH   = "world-of-shadow-work"

# --- HF token (REQUIRED for MODEL='flux'; FLUX.1-dev is GATED on Hugging Face). SDXL needs none. ---
# Accept the license at https://huggingface.co/black-forest-labs/FLUX.1-dev , paste a READ token.
HF_TOKEN = ""

# --- GALAXY CLIP model: LOCKED to the installed 50k galaxy (1024-d). DO NOT CHANGE. ---
# Confirmed: wosw_colab.ipynb cell 1 (MULTILINGUAL=True) + reagency/assets/manifest.json embed_dim=1024.
# This CLIP only EMBEDS dreams for galaxy placement; it is SEPARATE from the diffusion MODEL that
# GENERATES dream PNGs. Dreams MUST be embedded with THIS model or the merge is meaningless.
CLIP_MODEL      = "xlm-roberta-large-ViT-H-14"
CLIP_PRETRAINED = "frozen_laion5b_s13b_b90k"
CLIP_EXPECT_DIM = 1024          # hard-asserted before any embedding/bake; mismatch ABORTS the run.

# --- archival LoRA identity ---
TRIGGER = "wsw_archive"         # rare trigger token; injected into every train caption + every dream prompt
LORA_RANK     = {"sdxl": 32, "flux": 16}[MODEL]
LEARNING_RATE = {"sdxl": 1e-4, "flux": 1e-4}[MODEL]
LORA_STEPS    = {"sdxl": 1200, "flux": 2500}[MODEL]   # SDXL validate vs FLUX final
TRAIN_BATCH   = {"sdxl": 2,    "flux": 1}[MODEL]
TRAIN_RES     = 1024
TRAIN_SUBSET  = 1200            # cap LoRA train images (full ~14,759 is overkill for a style LoRA)
CAPTION_MODEL = "Salesforce/blip-image-captioning-large"   # BLIP auto-caption for the train set

# --- dream generation scope (hero selection) ---
N_MELT      = 8                 # op 1: img2img latent-blend dream-melt PAIRS
N_SHADOW    = 6                 # op 2: rembg/U2Net shadow inpaint (the disowned subject)
N_OUTPAINT  = 6                 # op 3: edge outpaint (extend past the frame)  -> 20 dreams total
MELT_STRENGTH   = 0.62          # img2img blend strength (0.55-0.70 keeps the archive legible)
INPAINT_STRENGTH = 0.92         # shadow/outpaint denoise on the masked region
GEN_RES     = 1024
HERO_SEED   = 42                # deterministic theme-balanced hero pool + melt pairing

# --- corpus reuse (same pattern as wosw_colab cell 5; avoids AIC 403 re-fetch) ---
USE_DRIVE_CORPUS = True
DRIVE_CORPUS     = "/content/drive/MyDrive/wosw/corpus.zip"   # a .zip OR a folder on Drive

# --- the EXISTING 50k galaxy work/ (embeddings.npy + meta.json) we MERGE against ---
# Prefer the REAL work/ from Drive (byte-identical to what baked the committed galaxy). Reconstructing
# via Stage A only reproduces the same 50k if the Drive corpus + concept words are identical.
DRIVE_WORK = "/content/drive/MyDrive/wosw/work"   # holds the pristine embeddings.npy + meta.json

# --- Drive CHECKPOINT root: every expensive stage lands here so a Colab reset doesn't lose work ---
DRIVE_CKPT = "/content/drive/MyDrive/wosw/dream_ckpt"   # captions/, lora_<MODEL>/, dreams/, work_ckpt/

# --- stage toggles (the BAKE DAG; each stage is independently resumable from Drive) ---
RUN_CAPTION = True   # BLIP-caption the LoRA train subset            (DAG step 1)
RUN_TRAIN   = True   # train the archival LoRA                       (DAG step 2, slow GPU stage)
RUN_DREAM   = True   # generate the 3 dream ops -> PNGs + provenance (DAG step 3)
RUN_REBAKE  = True   # embed dreams (galaxy CLIP) + concat + patch stage_b + re-run -> supersede (4-7)

# derived paths
MODEL_KEY = MODEL_ID
LORA_DIR  = os.path.join(DRIVE_CKPT, "lora_%s" % MODEL)     # trained adapter weights
DREAM_DIR = os.path.join(DRIVE_CKPT, "dreams")             # dream PNGs + provenance.json (CLIP-agnostic)
print("MODEL=%s (%s)  rank=%d  trigger=%r" % (MODEL, MODEL_KEY, LORA_RANK, TRIGGER))
print("galaxy CLIP=%s/%s  expect dim=%d  (SEPARATE from the diffusion model)" % (
    CLIP_MODEL, CLIP_PRETRAINED, CLIP_EXPECT_DIM))
if MODEL == "flux" and not HF_TOKEN:
    print("\n*** MODEL='flux' but HF_TOKEN is blank. FLUX.1-dev is HF-GATED. ***\n"
          "Accept the license at https://huggingface.co/black-forest-labs/FLUX.1-dev,\n"
          "then paste a READ token into HF_TOKEN above. Validate with MODEL='sdxl' FIRST.")

## GPU check

Mirrors `wosw_colab.ipynb` cell 2. FLUX.1-dev 12B requires the 80GB A100; the cell aborts FLUX on <70GB.
SDXL fits comfortably (~16GB).

In [ ]:
# ===== GPU check =====  (mirrors wosw_colab.ipynb cell 2)
!nvidia-smi -L
import torch
assert torch.cuda.is_available(), (
    "\n*** NO GPU — Runtime -> Change runtime type -> A100 (High-RAM) -> Save, then Run all. ***\n")
_name = torch.cuda.get_device_name(0)
_vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print("GPU: %s  |  VRAM: %.0f GB  |  torch %s" % (_name, _vram, torch.__version__))
if MODEL == "flux" and _vram < 70:
    raise SystemExit(
        "\n*** FLUX.1-dev 12B needs an 80GB A100; this card has %.0fGB. ***\n" % _vram +
        "Request an A100-80GB runtime, or set MODEL='sdxl' for the validation pass.\n")

## Clone the branch + mount Drive

Mirrors `wosw_colab.ipynb` cell 3. Clones `world-of-shadow-work`, `%cd`s into `reagency/factory` (so all
the factory scripts and the relative `../assets` path resolve exactly as in `wosw_colab.ipynb`), mounts
Drive, and creates the per-stage checkpoint dirs.

In [ ]:
# ===== clone branch + cd into the factory + mount Drive =====  (mirrors wosw_colab cell 3)
import os
_auth = (GH_TOKEN + "@") if GH_TOKEN else ""
if not os.path.isdir("MAT201B_Projects"):
    !git clone -b {BRANCH} https://{_auth}github.com/{REPO}.git
%cd MAT201B_Projects/reagency/factory
# Sanity: the factory scripts this notebook reuses must be present.
for _s in ("fetch_corpus.py", "stage_a_embed.py", "stage_b_layout.py"):
    assert os.path.exists(_s), "missing factory script: %s" % _s
from google.colab import drive
drive.mount("/content/drive")
for _p in (DRIVE_CKPT, LORA_DIR, DREAM_DIR,
           os.path.join(DRIVE_CKPT, "captions"), os.path.join(DRIVE_CKPT, "work_ckpt")):
    os.makedirs(_p, exist_ok=True)
os.makedirs("work", exist_ok=True)
print("factory @", os.getcwd())
print("checkpoint root:", DRIVE_CKPT)

## Pip installs

Two blocks, deduped: **Block 1** is the IDENTICAL galaxy stack from `wosw_colab.ipynb` cell 4 (Stage A
embed + Stage B layout reuse these — so dream embeddings land in the same 1024-d space). **Block 2** is
the diffusion / LoRA / segmentation stack (offline factory only; never ships in the allolib app).

In [ ]:
# ===== pip (hardened) =====
# Block 1: galaxy stages (Stage A + Stage B reuse these -- same set as wosw_colab cell 4).
!pip -q install open_clip_torch umap-learn faiss-cpu hdbscan scikit-learn plyfile pillow
# Block 2: diffusion / LoRA / segmentation (offline factory only; never ships in the app).
!pip -q install -U "diffusers>=0.32,<0.36" "transformers>=4.44" accelerate peft safetensors datasets sentencepiece
!pip -q install rembg onnxruntime
# FIX (torchao): Colab ships torchao 0.10, but peft's LoRA injection requires torchao>0.16 and
#   RAISES on the old one even though we never use it -> remove it so peft skips that dispatcher.
!pip -q uninstall -y torchao

# Verify the stack loads cleanly. If this throws a numpy "_center" ImportError, the -U installs
# half-upgraded numpy under the running kernel -> Runtime > Restart session, then Run all (one time).
import importlib.util as _u
import diffusers, peft, transformers
print("diffusers", diffusers.__version__, "| peft", peft.__version__,
      "| transformers", transformers.__version__,
      "| torchao removed:", _u.find_spec("torchao") is None)


## Corpus reuse + the existing 50k `work/` we merge against

Mirrors `wosw_colab.ipynb` cell 5 for the corpus (Drive `.zip`→unzip / folder→symlink, avoiding AIC 403).
**Also** restores the pristine 50k `work/embeddings.npy` + `work/meta.json` — the merge target — from
`DRIVE_WORK` (preferred: byte-identical to what baked the committed galaxy). If absent, reconstruct via
Stage A with the LOCKED galaxy CLIP (only reproduces the same 50k if the Drive corpus + concept words are
identical). The corpus images are the LoRA train set, the hero pool, and the provenance source.

In [ ]:
# ===== corpus reuse (wosw_colab cell 5) + restore the pristine 50k work/ (merge target) =====
import os, glob, shutil
# --- corpus (same logic as wosw_colab cell 5: zip -> unzip, folder -> symlink) ---
if USE_DRIVE_CORPUS and not os.path.isdir("corpus/images"):
    !rm -rf corpus _cz
    if DRIVE_CORPUS.endswith(".zip"):
        !unzip -q -o "{DRIVE_CORPUS}" -d _cz
        shutil.move("_cz/corpus" if os.path.isdir("_cz/corpus") else "_cz", "corpus")
    else:
        os.symlink(DRIVE_CORPUS, "corpus")
elif not os.path.isdir("corpus/images"):
    !python3 fetch_corpus.py     # last resort (AIC may 403); Drive reuse is strongly preferred
imgs = sorted(glob.glob("corpus/images/**/*.jpg", recursive=True))   # SAME glob as stage_a_embed.py:59
print("corpus images:", len(imgs), "| ATTRIBUTION.csv:", os.path.exists("corpus/ATTRIBUTION.csv"))
assert imgs, "empty corpus — set USE_DRIVE_CORPUS/DRIVE_CORPUS or run fetch_corpus.py"

# --- restore the pristine 50k work/ (the merge target) from Drive, or reconstruct via Stage A ---
if not os.path.exists("work/embeddings.npy"):
    if os.path.exists(os.path.join(DRIVE_WORK, "embeddings.npy")):
        shutil.copy(os.path.join(DRIVE_WORK, "embeddings.npy"), "work/embeddings.npy")
        shutil.copy(os.path.join(DRIVE_WORK, "meta.json"),       "work/meta.json")
        print("restored pristine work/ from", DRIVE_WORK)
    else:
        print("work/ ABSENT and no DRIVE_WORK -> reconstructing via Stage A with the LOCKED galaxy CLIP...")
        print("  (only reproduces the committed 50k if this corpus + concept words are byte-identical.)")
        !python3 stage_a_embed.py --model {CLIP_MODEL} --pretrained {CLIP_PRETRAINED} --corpus corpus --out work --batch 32

# --- idempotent pristine backup so the merge can always start from the clean 50k after a re-run ---
if not os.path.exists("work/embeddings.base50k.npy"):
    shutil.copy("work/embeddings.npy", "work/embeddings.base50k.npy")
    shutil.copy("work/meta.json",       "work/meta.base50k.json")
    print("saved pristine 50k backup: work/embeddings.base50k.npy + meta.base50k.json")

## GROUND assert — confirm the galaxy's embedding space (1024-d) BEFORE any bake

The project-killing bug is an embedding-space mismatch. We hard-assert the corpus embeddings are
`CLIP_EXPECT_DIM`=1024 (cross-checked against `assets/manifest.json embed_dim`) **before** any dream
embedding. Dreams will be embedded with the SAME `CLIP_MODEL`/`CLIP_PRETRAINED`. If a future galaxy was
baked with a different CLIP, this stops the run rather than silently producing a meaningless merge.

In [ ]:
# ===== GROUND assert: the merge target MUST be 1024-d, or the dream merge is meaningless =====
import json, numpy as np
E0 = np.load("work/embeddings.npy", mmap_mode="r")
meta0 = json.load(open("work/meta.json"))
print("corpus embeddings:", E0.shape, E0.dtype, "| meta records:", len(meta0))
assert E0.shape[1] == CLIP_EXPECT_DIM, (
    "embed_dim %d != galaxy %d. The corpus was embedded with a DIFFERENT CLIP than the installed "
    "galaxy. Re-embed Stage A with %s/%s." % (E0.shape[1], CLIP_EXPECT_DIM, CLIP_MODEL, CLIP_PRETRAINED))
assert E0.shape[0] == len(meta0), "embeddings/meta length mismatch (%d vs %d)" % (E0.shape[0], len(meta0))
# cross-check against the committed manifest if present in the cloned repo
_mani = "../assets/manifest.json"
if os.path.exists(_mani):
    _m = json.load(open(_mani))
    assert _m.get("embed_dim") == CLIP_EXPECT_DIM, "manifest embed_dim %s != %d" % (_m.get("embed_dim"), CLIP_EXPECT_DIM)
    print("manifest embed_dim:", _m.get("embed_dim"), "| n_points:", _m.get("n_points"))
N_CORPUS = E0.shape[0]
print("OK: %d corpus points x %d-d. Dreams will be embedded with the SAME model and APPENDED after these." % (
    N_CORPUS, CLIP_EXPECT_DIM))

## STEP 1 — CAPTION (archival-style, BLIP) → LoRA train set  ·  DAG step 1

Builds the LoRA training set from a theme-balanced `TRAIN_SUBSET` of `corpus/images/{agency,labor,magic}/*.jpg`.
Each caption = `"<TRIGGER>, <theme> archive, <BLIP caption>, museum public-domain photograph"`. The
trigger token anchors the archival style; the theme (read from the corpus subfolder — the same `theme`
Stage A stamps into meta, `stage_a_embed.py:84`) and the BLIP caption give the text encoder real content.

We write a HF `imagefolder` **`metadata.jsonl`** (the format both the SDXL and FLUX diffusers trainers
consume) and a **`caption_map.csv`** linking each train image → its real `source_id` (from
`corpus/ATTRIBUTION.csv`, keyed by basename exactly like `stage_a_embed.load_attrib`) so dream provenance
can point back at real artworks. Captioning is IDENTICAL for SDXL and FLUX — authored once, reused for
whichever `MODEL`. Drive-checkpointed so a reset doesn't force a re-caption.

*Reuses:* `corpus/images/{theme}/*.jpg` + `corpus/ATTRIBUTION.csv` (11-col schema from `fetch_corpus.py`).

In [ ]:
# ===== STEP 1: caption corpus -> work/train/ (imagefolder metadata.jsonl) =====
import os, glob, csv, json, random, shutil
TRAIN_ROOT = "work/train"
CAP_CKPT   = os.path.join(DRIVE_CKPT, "captions")

if RUN_CAPTION and not (os.path.exists(os.path.join(TRAIN_ROOT, "metadata.jsonl"))):
    # restore from Drive if a previous run captioned already
    if os.path.exists(os.path.join(CAP_CKPT, "metadata.jsonl")):
        os.makedirs(TRAIN_ROOT, exist_ok=True)
        !rsync -a "{CAP_CKPT}/" "{TRAIN_ROOT}/"
        print("restored caption set from Drive ->", TRAIN_ROOT)
if RUN_CAPTION and not os.path.exists(os.path.join(TRAIN_ROOT, "metadata.jsonl")):
    random.seed(HERO_SEED)
    os.makedirs(TRAIN_ROOT, exist_ok=True)
    # attribution: basename -> row (same key as stage_a_embed.load_attrib)
    ATTRIB = {}
    if os.path.exists("corpus/ATTRIBUTION.csv"):
        for r in csv.DictReader(open("corpus/ATTRIBUTION.csv", newline="")):
            ATTRIB[os.path.basename(r["file"])] = r
    # theme-balanced subset
    by_theme = {}
    for p in sorted(glob.glob("corpus/images/**/*.jpg", recursive=True)):
        by_theme.setdefault(os.path.basename(os.path.dirname(p)), []).append(p)
    per = max(1, TRAIN_SUBSET // max(1, len(by_theme)))
    train_imgs = []
    for th, ps in by_theme.items():
        random.shuffle(ps); train_imgs += ps[:per]
    random.shuffle(train_imgs); train_imgs = train_imgs[:TRAIN_SUBSET]
    # BLIP captioner (HF transformers; inference only). Greedy decode => deterministic re-runs.
    import torch
    from PIL import Image
    from transformers import BlipProcessor, BlipForConditionalGeneration
    proc = BlipProcessor.from_pretrained(CAPTION_MODEL)
    blip = BlipForConditionalGeneration.from_pretrained(CAPTION_MODEL).to("cuda").eval()
    def blip_caption(path):
        im = Image.open(path).convert("RGB")
        inp = proc(images=im, return_tensors="pt").to("cuda")
        with torch.no_grad():
            out = blip.generate(**inp, num_beams=1, max_new_tokens=24)
        return proc.decode(out[0], skip_special_tokens=True).strip()
    rows = []
    with open(os.path.join(TRAIN_ROOT, "metadata.jsonl"), "w") as mf:
        for k, p in enumerate(train_imgs):
            theme = os.path.basename(os.path.dirname(p))           # agency|labor|magic
            bn = os.path.basename(p)
            cap = blip_caption(p)
            text = "%s, %s archive, %s, museum public-domain photograph" % (TRIGGER, theme, cap)
            dst = "%05d.jpg" % k
            shutil.copy(p, os.path.join(TRAIN_ROOT, dst))           # imagefolder convention
            mf.write(json.dumps({"file_name": dst, "text": text}) + "\n")
            row = ATTRIB.get(bn, {})
            rows.append({"train_file": dst, "corpus_file": os.path.relpath(p, "corpus"),
                         "theme": theme, "source_id": row.get("source_id", ""),
                         "source": row.get("source", ""), "caption": text})
            if k % 100 == 0:
                print("  captioned %d/%d" % (k, len(train_imgs)))
    with open(os.path.join(TRAIN_ROOT, "caption_map.csv"), "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=["train_file","corpus_file","theme","source_id","source","caption"])
        w.writeheader(); w.writerows(rows)
    del blip, proc; torch.cuda.empty_cache()
    !rsync -a "{TRAIN_ROOT}/" "{CAP_CKPT}/"                           # Drive checkpoint
    print("captioned %d -> %s  (trigger=%r); checkpointed to Drive" % (len(rows), TRAIN_ROOT, TRIGGER))
else:
    n = sum(1 for _ in open(os.path.join(TRAIN_ROOT, "metadata.jsonl"))) if os.path.exists(os.path.join(TRAIN_ROOT, "metadata.jsonl")) else 0
    print("caption: using existing %s (%d rows)" % (TRAIN_ROOT, n) if n else "caption: SKIPPED (RUN_CAPTION=False)")

## STEP 2 — LoRA TRAIN  ·  DAG step 2 (the slow GPU stage)

Gated by `MODEL`. Uses diffusers' **official** example training scripts (fetched at the installed
diffusers version, falling back to `main`):
- `MODEL="sdxl"` → `examples/text_to_image/train_text_to_image_lora_sdxl.py` (sdxl-base-1.0 +
  `madebyollin/sdxl-vae-fp16-fix`, bf16, gradient checkpointing). **<1hr, ~16GB.** The fast pass that
  PROVES the whole DAG before paying for FLUX.
- `MODEL="flux"` → `examples/dreambooth/train_dreambooth_lora_flux.py` (FLUX.1-dev 12B, bf16, gradient
  checkpointing, `--cache_latents`). **HF-gated** (needs `HF_TOKEN`); uses most of the 80GB card; longer
  wall-clock than SDXL.

Weights → `LORA_DIR` on Drive (resume-friendly: `RUN_TRAIN=False` loads the existing adapter,
`--resume_from_checkpoint=latest` picks up a partial train). The two `MODEL` branches are mutually a
no-op.

In [ ]:
# ===== STEP 2: LoRA train (gated by MODEL) =====
import os, urllib.request, diffusers
_HAS_LORA = os.path.exists(LORA_DIR) and any(
    f.endswith(".safetensors") for f in os.listdir(LORA_DIR)) if os.path.isdir(LORA_DIR) else False

def _fetch_example(rel_path, local):
    """diffusers training scripts are NOT pip-installed; pull the version-matched example."""
    if os.path.exists(local):
        return
    ver = "v" + diffusers.__version__
    base = "https://raw.githubusercontent.com/huggingface/diffusers/%s/" + rel_path
    try:
        urllib.request.urlretrieve(base % ver, local)
    except Exception:
        urllib.request.urlretrieve(base % "main", local)

if RUN_TRAIN and not _HAS_LORA:
    os.makedirs(LORA_DIR, exist_ok=True)
    if MODEL == "sdxl":
        _fetch_example("examples/text_to_image/train_text_to_image_lora_sdxl.py",
                       "train_text_to_image_lora_sdxl.py")
        cmd = (
            "accelerate launch --num_processes=1 --mixed_precision=bf16 train_text_to_image_lora_sdxl.py "
            "--pretrained_model_name_or_path=%s "
            "--pretrained_vae_model_name_or_path=madebyollin/sdxl-vae-fp16-fix "
            "--train_data_dir=work/train --caption_column=text "
            "--resolution=%d --random_flip "
            "--train_batch_size=%d --gradient_accumulation_steps=2 "
            "--gradient_checkpointing "
            "--max_train_steps=%d --learning_rate=%g --lr_scheduler=constant --lr_warmup_steps=0 "
            "--rank=%d --checkpointing_steps=400 --resume_from_checkpoint=latest --seed=42 "
            "--output_dir=%s" % (MODEL_KEY, TRAIN_RES, TRAIN_BATCH, LORA_STEPS, LEARNING_RATE,
                                 LORA_RANK, LORA_DIR))
    else:  # flux
        assert HF_TOKEN, "FLUX.1-dev is HF-gated. Accept the license + set HF_TOKEN in CONFIG."
        from huggingface_hub import login, model_info; login(token=HF_TOKEN)
        try: model_info(MODEL_KEY, token=HF_TOKEN)
        except Exception as _e:
            raise SystemExit("FLUX.1-dev is gated: accept the license at "
                "https://huggingface.co/black-forest-labs/FLUX.1-dev with this token's account. (%s)" % _e)
        _fetch_example("examples/dreambooth/train_dreambooth_lora_flux.py",
                       "train_dreambooth_lora_flux.py")
        # FLUX --instance_data_dir Image.open()s EVERY file -> metadata.jsonl/caption_map.csv crash it.
        # Build an images-ONLY dir (FLUX dreambooth trains on the single --instance_prompt anyway).
        import glob as _g, shutil as _sh
        _tf = "work/train_flux"; os.makedirs(_tf, exist_ok=True)
        for _p in _g.glob("work/train/*.jpg"): _sh.copy(_p, _tf)
        cmd = (
            "accelerate launch --num_processes=1 --mixed_precision=bf16 train_dreambooth_lora_flux.py "
            "--pretrained_model_name_or_path=%s "
            "--instance_data_dir=work/train_flux "
            "--instance_prompt=\"%s, museum public-domain archive photograph\" "
            "--resolution=%d --train_batch_size=%d --gradient_accumulation_steps=4 "
            "--gradient_checkpointing --rank=%d --optimizer=adamw "
            "--max_train_steps=%d --learning_rate=%g --lr_scheduler=constant --lr_warmup_steps=0 "
            "--guidance_scale=1.0 --cache_latents --checkpointing_steps=500 "
            "--resume_from_checkpoint=latest --seed=42 "
            "--output_dir=%s" % (MODEL_KEY, TRIGGER, TRAIN_RES, TRAIN_BATCH, LORA_RANK,
                                 LORA_STEPS, LEARNING_RATE, LORA_DIR))
        print("NOTE: FLUX.1-dev 12B LoRA — uses most of the A100 80GB; longer wall-clock than SDXL.")
    print(cmd)
    !{cmd}
    print("%s LoRA -> %s | files: %s" % (MODEL, LORA_DIR, os.listdir(LORA_DIR)))
elif _HAS_LORA:
    print("RUN_TRAIN skip: existing %s LoRA at %s -> %s" % (MODEL, LORA_DIR, os.listdir(LORA_DIR)))
else:
    print("train: SKIPPED (RUN_TRAIN=False and no weights present)")

## STEP 3a — LOAD the trained pipeline (gated by MODEL)

One set of pipeline objects reused by all three dream ops. SDXL builds img2img + inpaint sharing the same
components (no second copy in VRAM, via `**pipe.components`); FLUX builds img2img + inpaint. The trained
LoRA adapter is loaded from `LORA_DIR` so every dream carries the archival style.

In [ ]:
# ===== STEP 3a: load pipeline + LoRA (gated by MODEL) =====
import torch
DTYPE = torch.bfloat16
if MODEL == "sdxl":
    from diffusers import (StableDiffusionXLImg2ImgPipeline,
                           StableDiffusionXLInpaintPipeline, AutoencoderKL)
    vae = AutoencoderKL.from_pretrained("madebyollin/sdxl-vae-fp16-fix", torch_dtype=DTYPE)
    pipe_i2i = StableDiffusionXLImg2ImgPipeline.from_pretrained(
        MODEL_KEY, vae=vae, torch_dtype=DTYPE).to("cuda")
    pipe_i2i.load_lora_weights(LORA_DIR)
    pipe_inp = StableDiffusionXLInpaintPipeline(**pipe_i2i.components).to("cuda")  # shares VRAM
    NEG = "low quality, modern, color photo, watermark, text, frame, border"
    GUID = 7.0
else:  # flux
    from huggingface_hub import login; login(token=HF_TOKEN)
    from diffusers import FluxImg2ImgPipeline, FluxInpaintPipeline
    pipe_i2i = FluxImg2ImgPipeline.from_pretrained(MODEL_KEY, torch_dtype=DTYPE).to("cuda")
    pipe_i2i.load_lora_weights(LORA_DIR)
    pipe_inp = FluxInpaintPipeline(**pipe_i2i.components).to("cuda")
    NEG  = None     # FLUX.1-dev is guidance-distilled; ignores negative prompts
    GUID = 3.5
pipe_out = pipe_inp  # edge-outpaint = inpaint on a padded canvas

def gen_kw(**kw):
    """backend-correct generation kwargs (SDXL takes negative_prompt; FLUX takes guidance_scale)."""
    if MODEL == "sdxl":
        kw["negative_prompt"] = NEG
    else:
        kw["guidance_scale"] = GUID
    return kw
print("pipelines ready for MODEL=%s; LoRA loaded from %s" % (MODEL, LORA_DIR))

## STEP 3b — HERO selection + provenance lookup + rembg

Heroes are the cluster representatives the galaxy already trusts: `work/cluster_reps.json` (written by
`stage_b_layout.py:289-302`, one highest-density image per cluster). If absent (fresh runtime), fall back
to a deterministic theme-balanced sample. We also build the **provenance lookup**: corpus relpath →
`(source_id, source, title, corpus_index)`, where `corpus_index` is the image's position in the
`sorted(corpus/images/**/*.jpg)` glob — i.e. its index in the preserved 50k order (image rows come
*first* in `meta`, `stage_a_embed.py:59-91`). And we load **rembg/U2Net** for the shadow-inpaint masks
(dependency-light vs SAM, reliable on Colab).

*Reuses:* `work/cluster_reps.json` schema `{cluster: {file,label,source_url}}`; `corpus/ATTRIBUTION.csv`
`source_id`.

In [ ]:
# ===== STEP 3b: heroes + provenance lookup + rembg =====
import os, json, glob, csv, random
random.seed(HERO_SEED)

# corpus relpath (e.g. 'images/labor/xxx.jpg') -> provenance, reusing ATTRIBUTION.csv (basename key)
ATTR_BY_BN = {}
if os.path.exists("corpus/ATTRIBUTION.csv"):
    for r in csv.DictReader(open("corpus/ATTRIBUTION.csv", newline="")):
        ATTR_BY_BN[os.path.basename(r["file"])] = r
# corpus_index == position in the SORTED image glob == index in the preserved 50k order (images first)
SORTED_IMGS = sorted(glob.glob("corpus/images/**/*.jpg", recursive=True))
REL2IDX = {os.path.relpath(p, "corpus"): i for i, p in enumerate(SORTED_IMGS)}

def prov_of(rel):
    """rel like 'images/labor/x.jpg' -> dict with source_id/source/title/corpus_index."""
    row = ATTR_BY_BN.get(os.path.basename(rel), {})
    return {"source_id": row.get("source_id", ""), "source": row.get("source", ""),
            "title": row.get("title", ""), "corpus_index": REL2IDX.get(rel, -1)}

def hero_pool():
    reps = "work/cluster_reps.json"
    if os.path.exists(reps):
        d = json.load(open(reps))
        files = [v["file"] for v in d.values() if os.path.exists(os.path.join("corpus", v["file"]))]
        if files:
            print("heroes from cluster_reps.json:", len(files)); return files
    pool = []
    for th in ("agency", "labor", "magic"):
        g = sorted(glob.glob("corpus/images/%s/*.jpg" % th))
        random.shuffle(g); pool += [os.path.relpath(p, "corpus") for p in g[:40]]
    print("heroes (fallback, no cluster_reps):", len(pool)); return pool

HEROES = hero_pool(); random.shuffle(HEROES)
assert len(HEROES) >= 2, "need >=2 heroes for dream-melt pairs"

# rembg / U2Net for the shadow mask (the 'disowned subject')
from rembg import remove, new_session
REMBG = new_session("u2net")
print("rembg ready; hero pool:", len(HEROES))

## STEP 3c — GENERATE the three dream ops → PNGs + provenance  ·  DAG step 3

The three Move-1 ops, each writing a **PNG** into `work/dreams/` (OUTSIDE `corpus/`, so the original 50k
order is untouched) and one provenance record `dream_id → {op, theme, source_indices[], source_ids[],
prompt}`:
1. **dream-melt** — `Image.blend` two heroes, then img2img at `MELT_STRENGTH` to fuse them. Provenance =
   **both** sources (two indices/ids).
2. **shadow inpaint** — rembg-segment the dominant subject, dilate the mask, inpaint that region AWAY (an
   empty room where the figure was). Provenance = the one source.
3. **edge outpaint** — paste the hero into a larger canvas, mask the outer ring, inpaint the continuation
   past the frame. Provenance = the one source.

Every prompt carries `TRIGGER` so the archival LoRA style applies. PNGs + `provenance.json` are
checkpointed to `DREAM_DIR` on Drive. `RUN_DREAM=False` reloads instead of regenerating. After this cell
we FREE the diffusion pipeline + rembg from VRAM before loading the heavy 1024-d CLIP.

In [ ]:
# ===== STEP 3c: generate the three dream ops -> work/dreams/*.png + provenance.json =====
import os, json, random
import numpy as np
from PIL import Image, ImageOps, ImageFilter
random.seed(HERO_SEED)
LOCAL_DREAMS = "work/dreams"
os.makedirs(LOCAL_DREAMS, exist_ok=True)
PROV_PATH = os.path.join(LOCAL_DREAMS, "provenance.json")

def _load(rel, size=GEN_RES):
    im = Image.open(os.path.join("corpus", rel)).convert("RGB")
    return ImageOps.fit(im, (size, size), Image.LANCZOS)

if RUN_DREAM and os.path.exists(os.path.join(DREAM_DIR, "provenance.json")) and not RUN_TRAIN:
    # reload from Drive checkpoint (only auto-reload when we didn't just retrain)
    !rsync -a "{DREAM_DIR}/" "{LOCAL_DREAMS}/"
    print("restored dreams from Drive ->", LOCAL_DREAMS)

if RUN_DREAM and not os.path.exists(PROV_PATH):
    g = torch.Generator("cuda").manual_seed(HERO_SEED)
    prov = {}
    # --- op 1: dream-melt (img2img latent-blend between two archive photos) ---
    for k in range(N_MELT):
        a, b = random.sample(HEROES, 2)
        ta, tb = a.split("/")[1], b.split("/")[1]
        base = Image.blend(_load(a), _load(b), 0.5)
        prompt = "%s, %s and %s archive merged, double-exposed museum photograph" % (TRIGGER, ta, tb)
        img = pipe_i2i(**gen_kw(prompt=prompt, image=base, strength=MELT_STRENGTH,
                                num_inference_steps=40, generator=g)).images[0]
        did = "melt_%03d" % k; img.save(os.path.join(LOCAL_DREAMS, did + ".png"))
        pa, pb = prov_of(a), prov_of(b)
        prov[did] = {"op": "dream-melt", "theme": "dream-melt", "prompt": prompt,
                     "sources": [a, b], "source_indices": [pa["corpus_index"], pb["corpus_index"]],
                     "source_ids": [pa["source_id"], pb["source_id"]],
                     "titles": [pa["title"], pb["title"]]}
    # --- op 2: shadow inpaint (rembg the subject -> inpaint it AWAY, the disowned) ---
    for k in range(N_SHADOW):
        a = random.choice(HEROES); th = a.split("/")[1]
        im = _load(a)
        cut = remove(im, session=REMBG)                       # RGBA; alpha = subject
        alpha = np.array(cut.split()[-1])
        mask = Image.fromarray((alpha > 16).astype("uint8") * 255).filter(ImageFilter.MaxFilter(15))
        mask = mask.filter(ImageFilter.GaussianBlur(4))       # dilate + feather the repaint region
        prompt = "%s, %s archive, empty room where the figure was, absence, museum photograph" % (TRIGGER, th)
        img = pipe_inp(**gen_kw(prompt=prompt, image=im, mask_image=mask, strength=INPAINT_STRENGTH,
                                num_inference_steps=40, generator=g)).images[0]
        did = "shadow_%03d" % k; img.save(os.path.join(LOCAL_DREAMS, did + ".png"))
        p = prov_of(a)
        prov[did] = {"op": "shadow-inpaint", "theme": "shadow-inpaint", "prompt": prompt,
                     "sources": [a], "source_indices": [p["corpus_index"]],
                     "source_ids": [p["source_id"]], "titles": [p["title"]]}
    # --- op 3: edge outpaint (pad canvas, inpaint the outer ring -> extend past the frame) ---
    for k in range(N_OUTPAINT):
        a = random.choice(HEROES); th = a.split("/")[1]
        core = _load(a, 768)
        canvas = Image.new("RGB", (GEN_RES, GEN_RES), (0, 0, 0)); canvas.paste(core, (128, 128))
        mask = Image.new("L", (GEN_RES, GEN_RES), 255)
        mask.paste(0, (128, 128, 128 + 768, 128 + 768))       # 0 = keep the original core
        prompt = "%s, %s archive extended beyond its frame, more of the scene, museum photograph" % (TRIGGER, th)
        img = pipe_out(**gen_kw(prompt=prompt, image=canvas, mask_image=mask, strength=INPAINT_STRENGTH,
                                num_inference_steps=40, generator=g)).images[0]
        did = "outpaint_%03d" % k; img.save(os.path.join(LOCAL_DREAMS, did + ".png"))
        p = prov_of(a)
        prov[did] = {"op": "edge-outpaint", "theme": "edge-outpaint", "prompt": prompt,
                     "sources": [a], "source_indices": [p["corpus_index"]],
                     "source_ids": [p["source_id"]], "titles": [p["title"]]}
    json.dump(prov, open(PROV_PATH, "w"), indent=2)
    os.makedirs(DREAM_DIR, exist_ok=True)
    !rsync -a "{LOCAL_DREAMS}/" "{DREAM_DIR}/"                  # Drive checkpoint
    print("generated %d dreams -> %s (+ provenance.json); checkpointed to Drive" % (len(prov), LOCAL_DREAMS))
else:
    n = len(json.load(open(PROV_PATH))) if os.path.exists(PROV_PATH) else 0
    print("dreams: using existing %s (%d dreams)" % (LOCAL_DREAMS, n) if n else "dreams: SKIPPED (RUN_DREAM=False)")

# free diffusion + rembg VRAM BEFORE loading the heavy 1024-d CLIP (avoids OOM, esp. on the FLUX pass)
for _v in ("pipe_i2i", "pipe_inp", "pipe_out", "vae", "REMBG"):
    globals().pop(_v, None)
import gc; gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## STEP 4 — RE-EMBED dreams with the GALAXY's CLIP + CONCATENATE (append)  ·  DAG steps 4-5

**The load-bearing cell.** We embed ONLY the dream PNGs with the IDENTICAL galaxy model
(`xlm-roberta-large-ViT-H-14` / `frozen_laion5b_s13b_b90k`, 1024-d, L2-normalized, float32 — exactly the
`stage_a_embed.py:54/76-78` recipe), then **append** them after the pristine 50k (`E0`/`meta0` from
`work/embeddings.base50k.npy` — so the original order/ids never move), and **overwrite** `work/` so the
unchanged-invocation `stage_b_layout.py` re-bakes the COMBINED set.

Dream meta records use `type='dream'` (a STRING — `stage_b` checks `type=='word'`; the int **2** is
minted only in `points.bin` by the patch in STEP 5) and carry full provenance (`source_indices` into the
preserved 50k order + museum `source_ids`, supporting 2 sources for melts). We also DELETE
`work/coords.npy` so STEP 6 runs a FRESH UMAP over the enlarged set (a stale cache would lay dreams out
on the OLD geometry, `stage_b_layout.py:259-261`).

*Reuses:* `stage_a_embed.py` embedding recipe + meta schema; GROUND merge strategy (append, not in-corpus).

In [ ]:
# ===== STEP 4: embed dreams with the galaxy CLIP, APPEND to the pristine 50k, overwrite work/ =====
import os, json, glob, shutil
import numpy as np

if RUN_REBAKE:
    # always rebuild the merge from the PRISTINE 50k backup so re-runs are idempotent
    E0 = np.load("work/embeddings.base50k.npy")
    meta0 = json.load(open("work/meta.base50k.json"))
    assert E0.shape[1] == CLIP_EXPECT_DIM, "baseline not %d-d" % CLIP_EXPECT_DIM
    N_corpus = E0.shape[0]
    prov = json.load(open(os.path.join(LOCAL_DREAMS, "provenance.json")))
    dream_pngs = sorted(glob.glob(os.path.join(LOCAL_DREAMS, "*.png")))
    assert dream_pngs, "no dream PNGs in %s — run STEP 3c" % LOCAL_DREAMS

    # embed dream PNGs with the IDENTICAL galaxy model (stage_a_embed.py recipe)
    import torch, open_clip
    from PIL import Image
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    model, _, preprocess = open_clip.create_model_and_transforms(CLIP_MODEL, pretrained=CLIP_PRETRAINED)
    model = model.to(dev).eval()
    embs, meta_dreams = [], []
    with torch.no_grad():
        for i in range(0, len(dream_pngs), 32):
            batch = dream_pngs[i:i + 32]
            ims = [preprocess(Image.open(p).convert("RGB")) for p in batch]
            f = model.encode_image(torch.stack(ims).to(dev))
            f = f / f.norm(dim=-1, keepdim=True)                 # L2-normalize (stage_a_embed.py:77)
            embs.append(f.cpu().numpy().astype(np.float32))       # float32 (stage_a_embed.py:78)
            for p in batch:
                did = os.path.splitext(os.path.basename(p))[0]
                r = prov.get(did, {})
                meta_dreams.append({
                    "type": "dream",                              # -> points.bin type=2 (patched stage_b)
                    "theme": r.get("theme", "dream"),             # dream-melt|shadow-inpaint|edge-outpaint
                    "file": os.path.relpath(p, "corpus") if False else "work/dreams/%s.png" % did,
                    "label": did,
                    "creator": "WSW dream (%s LoRA)" % MODEL,
                    "source": "dream",
                    "license": "derived: CC0/public-domain archive",
                    "source_url": "",
                    "dream_id": did,
                    "source_indices": r.get("source_indices", []),   # into the preserved 50k order
                    "source_ids": r.get("source_ids", []),           # museum ids (HumanTrace key)
                    "prompt": r.get("prompt", ""),
                })
    del model; gc = __import__("gc"); gc.collect(); torch.cuda.empty_cache()
    Ed = np.concatenate(embs, axis=0).astype(np.float32)
    assert Ed.shape[1] == CLIP_EXPECT_DIM, "dream dim %d != galaxy %d (WRONG CLIP MODEL)" % (Ed.shape[1], CLIP_EXPECT_DIM)

    # SANITY: dreams should land NEAR their source artworks; a far-off dream = LoRA garbage.
    _bad = 0
    for _j, _m in enumerate(meta_dreams):
        _si = _m.get('source_indices', [])
        if _si:
            _mu = E0[_si].mean(0); _c = float(Ed[_j] @ _mu / (np.linalg.norm(_mu) + 1e-9))
            if _c < 0.15: _bad += 1
    if _bad: print('  WARNING: %d/%d dreams land far (<0.15 cos) from their sources -- check the LoRA output' % (_bad, len(meta_dreams)))

    # APPEND: 50k FIRST, dreams after -> original order/ids preserved (GROUND merge strategy)
    E_comb = np.concatenate([E0, Ed], axis=0).astype(np.float32)
    meta_comb = list(meta0) + meta_dreams
    np.save("work/embeddings.npy", E_comb)
    json.dump(meta_comb, open("work/meta.json", "w"))
    if os.path.exists("work/coords.npy"):
        os.remove("work/coords.npy")                              # force a FRESH UMAP over the combined set
    # Drive checkpoint the merged set
    np.save(os.path.join(DRIVE_CKPT, "work_ckpt", "embeddings.combined.npy"), E_comb)
    json.dump(meta_comb, open(os.path.join(DRIVE_CKPT, "work_ckpt", "meta.combined.json"), "w"))
    print("merged: %d corpus + %d dreams = %d points. UMAP cache cleared; checkpointed to Drive." % (
        N_corpus, Ed.shape[0], E_comb.shape[0]))
else:
    print("merge: SKIPPED (RUN_REBAKE=False)")

## STEP 5 — PATCH stage_b so dreams emit `type=2`  ·  DAG step 6

`stage_b_layout.py:266-267` computes `is_word = [type=='word']` then `ptype = is_word.astype(float32)` —
so it emits ONLY `type` 0 (image) / 1 (word). Our `type='dream'` records would otherwise bake as
`type=0`. We apply an **idempotent in-notebook string-patch** to `stage_b_layout.py` that adds an
`is_dream` mask and makes `ptype = where(is_dream, 2.0, is_word)` — verified to match the file verbatim,
stay AST-valid, and define `is_dream` before its use. We also tint dreams distinctly in `colorize`. The
`assert src != orig` guard catches a double-run (anchors already consumed) instead of silently
mis-patching. `build_atlas` (gates on `type=='image'`) and `build_labels` (gates on `type=='word'`)
leave dreams out by design — the atlas is the *real-archive* human trace; dreams are visible as colored
galaxy points but never as hero thumbnails.

> After a Colab reset, re-clone (restores the unpatched script) before re-running this cell — the patch
> is one-shot.

In [ ]:
# ===== STEP 5: idempotent patch -- stage_b emits type=2 for dreams + tints them + honest manifest =====
if RUN_REBAKE:
    orig = open("stage_b_layout.py").read()
    if "is_dream" in orig:
        print("stage_b_layout.py already patched (is_dream present) -- skipping (idempotent).")
    else:
        src = orig
        # (1) type encoding: image=0 / word=1 / dream=2  (anchor matches stage_b_layout.py:266-267 verbatim)
        src = src.replace(
            '    is_word = np.array([m["type"] == "word" for m in meta])\n'
            '    ptype = is_word.astype(np.float32)\n',
            '    is_word = np.array([m["type"] == "word" for m in meta])\n'
            '    is_dream = np.array([m["type"] == "dream" for m in meta])\n'
            '    ptype = np.where(is_dream, 2.0, is_word.astype(np.float32)).astype(np.float32)\n'
            '    print("[stage_b] %d dream points (type=2)" % int(is_dream.sum()))\n')
        # (2) distinct tint for dreams: warm/violet so they read as their own stratum in the galaxy
        src = src.replace(
            '    colors = colorize(cluster_id, density, is_word)\n',
            '    colors = colorize(cluster_id, density, is_word)\n'
            '    for _i in range(len(colors)):\n'
            '        if is_dream[_i]:\n'
            '            colors[_i] = (0.95, 0.45 + 0.40 * float(density[_i]), 0.85)  # dream = violet/warm\n')
        # (3) honest manifest: don't count dreams as images; add n_dreams
        src = src.replace(
            "'n_images': int((~is_word).sum())",
            "'n_images': int((~is_word & ~is_dream).sum()), 'n_dreams': int(is_dream.sum())")
        assert "is_dream" in src, ("STEP 5 patch anchors changed vs stage_b_layout.py -- "
                                   "re-derive the .replace targets.")
        import ast; ast.parse(src)                      # must AST-parse before we trust it
        open("stage_b_layout.py", "w").write(src)
        print("patched stage_b_layout.py: type=2 + dream tint + honest manifest (AST-valid).")
else:
    print("patch: SKIPPED (RUN_REBAKE=False)")


## STEP 6 — RE-BAKE the COMBINED galaxy + VERIFY type=2  ·  DAG step 7

Run `stage_b_layout.py` with the **standard `wosw_colab` invocation** (it reads the overwritten `work/`
and writes `../assets`, superseding the committed galaxy). UMAP runs fresh over the ~50,681-point set
(cache deleted in STEP 4). Then we read back `../assets/points.bin` (WSWP: `[magic][i32 ver,N,stride=10]`
then `N*10 float32`, type at column 8) and **hard-assert** the dream points landed as `type=2`. This is
the make-or-break verification — if the patch or merge failed, the count is 0 and the cell raises.

In [ ]:
# ===== STEP 6: re-bake + verify type=2 dream points in points.bin =====
import json, struct
import numpy as np
if RUN_REBAKE:
    !python3 stage_b_layout.py     # default args == wosw_colab cell 8 (--work work --assets ../assets ...)
    meta = json.load(open("work/meta.json"))
    n_dream_meta = sum(1 for m in meta if m.get("type") == "dream")
    with open("../assets/points.bin", "rb") as f:
        assert f.read(4) == b"WSWP", "bad points.bin magic"
        ver, N, stride = struct.unpack("<iii", f.read(12))
        arr = np.frombuffer(f.read(N * stride * 4), dtype="<f4").reshape(N, stride)
    t = arr[:, 8]
    hist = {int(v): int((t == v).sum()) for v in np.unique(t)}
    print("re-baked points.bin: N=%d | type0(img)=%d type1(word)=%d type2(dream)=%d" % (
        N, hist.get(0, 0), hist.get(1, 0), hist.get(2, 0)))
    assert hist.get(2, 0) == n_dream_meta and n_dream_meta > 0, (
        "type=2 dream count %d != meta dreams %d — patch/merge failed" % (hist.get(2, 0), n_dream_meta))
    print("new manifest:", json.load(open("../assets/manifest.json")))
else:
    print("re-bake: SKIPPED (RUN_REBAKE=False)")

## STEP 7 — re-key provenance for the runtime  ·  HumanTrace

The runtime addresses galaxy points by their `points.bin` row index. Dreams are appended after the 50k
corpus, so dream `point_index = N_corpus + k`. We write `../assets/dream_provenance.json` keyed by that
runtime point index → `{op, theme, source_indices[], source_ids[], titles[], prompt}` so HumanTrace can,
from a clicked dream point, surface the *real* source artworks (split-on-list, supporting the two sources
of a dream-melt). `source_indices` index into the preserved 50k order; `source_ids` are the durable
museum keys from `ATTRIBUTION.csv`.

In [ ]:
# ===== STEP 7: emit assets/dream_provenance.json keyed by runtime point index =====
import json
if RUN_REBAKE:
    meta = json.load(open("work/meta.json"))
    runtime_prov = {}
    for idx, m in enumerate(meta):
        if m.get("type") == "dream":
            runtime_prov[str(idx)] = {
                "dream_id": m.get("dream_id", m.get("label", "")),
                "op": m.get("theme", "dream"),
                "theme": m.get("theme", "dream"),
                "source_indices": m.get("source_indices", []),
                "source_ids": m.get("source_ids", []),
                "prompt": m.get("prompt", ""),
                "model": MODEL,
            }
    json.dump(runtime_prov, open("../assets/dream_provenance.json", "w"), indent=2)
    print("wrote ../assets/dream_provenance.json: %d dream points (keyed by runtime point index)" % len(runtime_prov))
else:
    print("provenance: SKIPPED (RUN_REBAKE=False)")

## STEP 8 — CHECKPOINT + PUSH the superseding galaxy

Mirror the re-baked assets to Drive (survives a Colab reset), then push to `world-of-shadow-work` so the
Mac `git pull` picks up the NEW galaxy (mirrors `wosw_colab.ipynb` cell 13). The committed 50k galaxy is
superseded; cluster/hero/atlas indices have shifted (accepted — the commit message flags it). The dream
PNGs themselves live in `work/dreams/` + on Drive (not committed — the runtime needs only the baked
`.bin`/`.png`/`.json` assets). No `GH_TOKEN` → download a tarball.

In [ ]:
# ===== STEP 8: checkpoint to Drive + DOWNLOAD the re-baked galaxy to the Mac (NO git push) =====
# PROJECT RULE (per the artist): WOSW factory assets DOWNLOAD to the Mac; never git-push from Colab.
# You verify the tarball locally, install into reagency/assets/, and commit/push from the Mac only.
import os, shutil
if RUN_REBAKE:
    _changed = ("points.bin", "edges.bin", "manifest.json", "atlas_0.png", "labels.txt",
                "labels_atlas.png", "label_words.txt", "classify.txt", "dream_provenance.json")
    _ad = os.path.join(DRIVE_CKPT, "assets"); os.makedirs(_ad, exist_ok=True)
    _files = []
    for fn in _changed:
        src = os.path.join("..", "assets", fn)
        if os.path.exists(src):
            shutil.copy(src, os.path.join(_ad, fn))   # Drive checkpoint
            _files.append("assets/" + fn)
    print("checkpointed re-baked assets ->", _ad)
    # tar ONLY the changed galaxy files (NOT the gitignored 57MB vessel, NOT the unchanged audio)
    _arglist = " ".join(_files)
    !tar czf /content/wsw_assets_dreams.tgz -C .. {_arglist}
    !ls -lh /content/wsw_assets_dreams.tgz
    from google.colab import files
    files.download("/content/wsw_assets_dreams.tgz")
    print("\nDOWNLOADED wsw_assets_dreams.tgz. On your Mac:")
    print("  tar xzf ~/Downloads/wsw_assets_dreams.tgz -C MAT201B_Projects/reagency/   # installs assets/*")
    print("  (review -> this SUPERSEDES the committed galaxy; commit/push from the Mac when ready)")
else:
    print("download: SKIPPED (RUN_REBAKE=False)")


## Run-order notes — the two-pass workflow

**Pass 1 — SDXL validate (do this FIRST, end-to-end).** In CONFIG set `MODEL="sdxl"` (no `HF_TOKEN`
needed), `RUN_CAPTION=RUN_TRAIN=RUN_DREAM=RUN_REBAKE=True`. Runtime → A100 (any 16GB+ is fine for SDXL) →
**Run all**. This proves the whole BAKE DAG cheaply (<1hr LoRA): caption → train → 20 dreams → embed →
append → patch stage_b → re-bake. The hard checks in STEP 6 (`type2 == n_dream_meta > 0`) and STEP 4
(append-not-insert, 1024-d) are your green light. Leave `GH_TOKEN` blank on this pass to download/inspect
the tarball rather than overwrite the committed galaxy.

**Pass 2 — FLUX final bake.** Accept the FLUX.1-dev license on Hugging Face, paste a READ token into
`HF_TOKEN`, set `MODEL="flux"`, request an **A100 80GB** runtime. Run all. The SDXL/FLUX cells are
mutually no-ops, so only the FLUX branch runs; the merge/patch/re-bake stages are model-agnostic. Set
`GH_TOKEN` to push the final superseding galaxy.

**Resumability (each stage is independently checkpointed to Drive under `DRIVE_CKPT`).** After a Colab
reset, re-run the setup cells (clone restores the *unpatched* `stage_b_layout.py` — required because the
STEP 5 patch is one-shot), then set the `RUN_*` flag of any already-checkpointed stage to `False` to
reload instead of recompute: `RUN_CAPTION=False` (reloads `captions/`), `RUN_TRAIN=False` (reloads
`lora_<MODEL>/`), `RUN_DREAM=False` (reloads `dreams/`). The merge always rebuilds from the pristine
`work/embeddings.base50k.npy`, so STEP 4 is idempotent. Switching `MODEL` between passes uses a separate
`LORA_DIR` (`lora_sdxl` vs `lora_flux`), so the two trains never collide.

**Caveats.** Re-baking SUPERSEDES the committed galaxy — cluster ids, hero/atlas slots, and ALL point
indices shift (accepted). UMAP omits `random_state` at 50k+ (`stage_b_layout.py:36`) so each re-bake's 3D
layout differs slightly even at seed 42; only the high-D embeddings/edges are reproducible. Dreams are
PNGs in `work/dreams/` (OUTSIDE `corpus/`) so the `.jpg`-only Stage A glob and the 50k ordering are never
disturbed — never move them into `corpus/images/`.